# 💳 Credit Card Fraud Detection
### Task 5 - Machine Learning Project

**Objective:** Build a machine learning model to detect fraudulent credit card transactions.

**Steps:**
1. Load and explore the dataset (EDA)
2. Preprocess and normalize data
3. Handle class imbalance
4. Train classification models
5. Evaluate using Precision, Recall, F1-Score

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

print('All libraries imported successfully!')

## 2. Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../data/creditcard.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic information
print('Dataset Info:')
print(df.info())
print('\nMissing values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
df.describe()

In [ ]:
# Class distribution
fraud_count = df['Class'].value_counts()
fraud_pct = df['Class'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(f'  Genuine (0): {fraud_count[0]:,} ({fraud_pct[0]:.2f}%)')
print(f'  Fraud   (1): {fraud_count[1]:,} ({fraud_pct[1]:.2f}%)')

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Genuine', 'Fraud'], fraud_count.values, color=['steelblue', 'crimson'])
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Count')

axes[1].pie(fraud_count.values, labels=['Genuine', 'Fraud'],
            autopct='%1.2f%%', colors=['steelblue', 'crimson'])
axes[1].set_title('Class Distribution (Percentage)')

plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Transaction Amount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

genuine = df[df['Class'] == 0]['Amount']
fraud = df[df['Class'] == 1]['Amount']

axes[0].hist(genuine, bins=50, color='steelblue', alpha=0.7, label='Genuine')
axes[0].hist(fraud, bins=50, color='crimson', alpha=0.7, label='Fraud')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].boxplot([genuine, fraud], labels=['Genuine', 'Fraud'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Amount Boxplot by Class')
axes[1].set_ylabel('Amount')

plt.tight_layout()
plt.savefig('../outputs/amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Avg Genuine Amount: ${genuine.mean():.2f}')
print(f'Avg Fraud Amount:   ${fraud.mean():.2f}')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(16, 10))
corr = df.corr()
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.5, fmt='.1f')
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# Scale 'Amount' and 'Time' columns
scaler = StandardScaler()

df['scaled_Amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_Time']   = scaler.fit_transform(df[['Time']])

# Drop original columns
df_processed = df.drop(['Amount', 'Time'], axis=1)

print('Preprocessing done!')
print('New shape:', df_processed.shape)
df_processed.head()

## 5. Train-Test Split

In [ ]:
X = df_processed.drop('Class', axis=1)
y = df_processed['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:   {X_train.shape[0]:,} samples')
print(f'Test set:       {X_test.shape[0]:,} samples')
print(f'Train Fraud %:  {y_train.mean()*100:.2f}%')
print(f'Test Fraud %:   {y_test.mean()*100:.2f}%')

## 6. Handle Class Imbalance (SMOTE)

In [ ]:
print('Before SMOTE:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(pd.Series(y_train_res).value_counts())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pd.Series(y_train).value_counts().plot(kind='bar', ax=axes[0],
    color=['steelblue','crimson'], title='Before SMOTE')
pd.Series(y_train_res).value_counts().plot(kind='bar', ax=axes[1],
    color=['steelblue','crimson'], title='After SMOTE')
plt.tight_layout()
plt.savefig('../outputs/smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Training

In [ ]:
# --- Model 1: Logistic Regression ---
print('Training Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_res, y_train_res)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
print('Done!')

In [ ]:
# --- Model 2: Random Forest ---
print('Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_res, y_train_res)
rf_preds = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
print('Done!')

## 8. Model Evaluation

In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba):
    print(f'\n===== {name} =====')
    print(classification_report(y_true, y_pred, target_names=['Genuine', 'Fraud']))
    print(f'ROC-AUC Score: {roc_auc_score(y_true, y_proba):.4f}')
    return {
        'Model': name,
        'Precision': precision_score(y_true, y_pred),
        'Recall':    recall_score(y_true, y_pred),
        'F1-Score':  f1_score(y_true, y_pred),
        'ROC-AUC':   roc_auc_score(y_true, y_proba)
    }

results = []
results.append(evaluate_model('Logistic Regression', y_test, lr_preds, lr_proba))
results.append(evaluate_model('Random Forest',       y_test, rf_preds, rf_proba))

results_df = pd.DataFrame(results)
print('\n===== Summary Table =====')
print(results_df.to_string(index=False))

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, name in zip(axes,
    [lr_preds, rf_preds],
    ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Genuine', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)

plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))

for proba, name, color in [
    (lr_proba, 'Logistic Regression', 'steelblue'),
    (rf_proba, 'Random Forest', 'crimson')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=color, lw=2)

plt.plot([0,1],[0,1],'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance (Random Forest)
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns)
top_features = feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Conclusion

| Model | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|
| Logistic Regression | ~0.XX | ~0.XX | ~0.XX | ~0.XX |
| Random Forest | ~0.XX | ~0.XX | ~0.XX | ~0.XX |

**Key Findings:**
- The dataset is highly imbalanced (~0.17% fraud). SMOTE helped balance the training data.
- Random Forest generally outperforms Logistic Regression on this task.
- For fraud detection, **Recall** is the most critical metric (we want to catch all frauds).
- ROC-AUC scores close to 1.0 indicate excellent model discrimination ability.